In [1]:
import pandas as pd

train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
sample_sub = pd.read_csv('SampleSubmission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print()
print(train.head(10))
print()
print("Unique subsets/languages:")
print(train['subset'].value_counts())

Train shape: (29815, 4)
Test shape: (2618, 3)

                       ID                                              input  \
0  ID_TR_Aka_Gha_A3B1799D  Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a...   
1  ID_TR_Aka_Gha_1C80317F  Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu...   
2  ID_TR_Aka_Gha_06671AD1  Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔ...   
3  ID_TR_Aka_Gha_BDD640FB  Dɛn ne aduru a wodi si nyisɛn ano ntɛm ntɛm, n...   
4  ID_TR_Aka_Gha_46685257  Hu sɛnea ɛyɛ den sɛ wubehu bɔbea mu basabasayɔ...   
5  ID_TR_Aka_Gha_3EB13B90  ɛdɛn haw ne nsusuaso aa ani nndaso nya wɔ nea ...   
6  ID_TR_Aka_Gha_392456DE  So ayaresa ahorow wɔ hɔ a wobetumi de adi dwum...   
7  ID_TR_Aka_Gha_23B8C1E9  Akwan bɛn na akyerɛkyerɛfo betumi afa so adi a...   
8  ID_TR_Aka_Gha_BC8960A9  Dwuma bɛn na mpanyimfo a wotumi de wɔn ho to w...   
9  ID_TR_Aka_Gha_1A3D1FA7  Na sɛ minni akwahosan ho insurance nso ɛ, so a...   

                                              output   subset  
0  Mmabu

In [2]:
# Check answer/question lengths
train['input_len'] = train['input'].str.split().str.len()
train['output_len'] = train['output'].str.split().str.len()

print(train.groupby('subset')[['input_len', 'output_len']].mean())
print()
print("Test set subset distribution:")
print(test['subset'].value_counts())
print()
print("Sample submission columns:")
print(sample_sub.columns.tolist())
print(sample_sub.head())

         input_len  output_len
subset                        
Aka_Gha  28.829181  105.632323
Amh_Eth   9.521951   20.231436
Eng_Eth  12.224266   24.471009
Eng_Gha  20.045240   75.117713
Eng_Ken  11.444712   78.699519
Eng_Uga  11.245016   95.383001
Lug_Uga  10.536210   79.682530
Swa_Ken  11.234300   84.291787

Test set subset distribution:
subset
Eng_Uga    744
Aka_Gha    492
Eng_Gha    491
Lug_Uga    374
Swa_Ken    229
Eng_Ken    167
Amh_Eth     61
Eng_Eth     60
Name: count, dtype: int64

Sample submission columns:
['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
                       ID             TargetRLF1             TargetR1F1  \
0  ID_TS_Aka_Gha_A3B1799D  Wuna dey craze, eweeh  Wuna dey craze, eweeh   
1  ID_TS_Aka_Gha_1C80317F  Wuna dey craze, eweeh  Wuna dey craze, eweeh   
2  ID_TS_Aka_Gha_06671AD1  Wuna dey craze, eweeh  Wuna dey craze, eweeh   
3  ID_TS_Aka_Gha_BDD640FB  Wuna dey craze, eweeh  Wuna dey craze, eweeh   
4  ID_TS_Aka_Gha_46685257  Wuna dey craze, eweeh  Wuna 

In [3]:
!pip install transformers datasets sentencepiece accelerate -q

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

# We'll add the language as a prefix to the input - this helps a multilingual
# model know what language to generate the answer in
train['input_text'] = "answer in " + train['subset'] + ": " + train['input']
test['input_text'] = "answer in " + test['subset'] + ": " + test['input']

# Split train into train/validation so we can track performance during training
train_df, val_df = train_test_split(train, test_size=0.1, random_state=42, stratify=train['subset'])

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print()
print(train_df[['input_text', 'output']].head(3))

Train size: 26833
Val size: 2982

                                              input_text  \
10480  answer in Eng_Gha: What are the potential long...   
419    answer in Aka_Gha: Nnuro bɛn saa na wɔde yɛ 's...   
28735  answer in Swa_Ken: Je, ni lazima kupima Ukimwi...   

                                                  output  
10480  Increased Risk of Mental Health Disorders: Pro...  
419    Nsa, nnuro a ɛma nna (sedatives) ma woda, w'an...  
28735  Kwa hakika si lazima kupima Ukimwi. Kuchagua k...  


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loaded successfully")
print(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded successfully
T5Tokenizer(name_or_path='google/mt5-small', vocab_size=250100, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250000: AddedToken("▁<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250001: AddedToken("▁<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250002: AddedToken("▁<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250003: AddedToken("▁<extra_id_96>", rstrip=F

In [6]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[['input_text', 'output']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['input_text', 'output']].reset_index(drop=True))

max_input_len = 128
max_output_len = 128

def preprocess(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=max_input_len,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        examples['output'],
        max_length=max_output_len,
        truncation=True,
        padding='max_length'
    )
    labels['input_ids'] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

print("Tokenization done")
print(train_tokenized)

Map:   0%|          | 0/26833 [00:00<?, ? examples/s]

Map:   0%|          | 0/2982 [00:00<?, ? examples/s]

Tokenization done
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26833
})


In [7]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-healthqa-v1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer set up. Starting training...")
trainer.train()

Trainer set up. Starting training...


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10065, training_loss=0.0, metrics={'train_runtime': 2236.6332, 'train_samples_per_second': 35.991, 'train_steps_per_second': 4.5, 'total_flos': 1.064095534153728e+16, 'train_loss': 0.0, 'epoch': 3.0})

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Reload a fresh, untrained model since the previous one is now broken/NaN
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-healthqa-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    logging_steps=100,
    save_total_limit=1,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer set up (fp16 disabled). Starting training...")
trainer.train()

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Trainer set up (fp16 disabled). Starting training...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,2.605705,2.126633
2,2.232218,1.908628
3,2.144199,1.840160


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10065, training_loss=2.5783054948741384, metrics={'train_runtime': 5082.7529, 'train_samples_per_second': 15.838, 'train_steps_per_second': 1.98, 'total_flos': 1.064095534153728e+16, 'train_loss': 2.5783054948741384, 'epoch': 3.0})

In [9]:
import torch

def generate_predictions(model, tokenizer, texts, max_length=128, batch_size=16):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    predictions = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded)
        if i % 160 == 0:
            print(f"Processed {i}/{len(texts)}")

    return predictions

# Quick sanity check on a few validation examples first
sample_inputs = val_df['input_text'].tolist()[:5]
sample_outputs = generate_predictions(model, tokenizer, sample_inputs)

for inp, pred, true in zip(sample_inputs, sample_outputs, val_df['output'].tolist()[:5]):
    print("INPUT:", inp[:80])
    print("PREDICTED:", pred)
    print("TRUE:", true[:150])
    print("---")

Processed 0/5
INPUT: answer in Eng_Uga: How do needle exchange programs help reduce HIV?
PREDICTED: This is a question about, HIV/AIDS. HIV is a virus that attacks the immune system, which attacks the immune system, and weakens the immune system. HIV is a virus that attacks the immune system, which weakens the immune system, and weakens the immune system. HIV is a virus that attacks the immune system, which weakens the immune system, and weakens the immune system. HIV is a virus that attacks the immune system, and weakens the immune system
TRUE: Needle-exchange programs have been widely demonstrated to be effective in reducing substance use-related risk behaviors (such as needle sharing) witho
---
INPUT: answer in Eng_Ken: How Long Does It Take for HIV to Cause AIDS?
PREDICTED: HIV (Human Immunodeficiency Virus) is a virus that attacks the immune system, specifically CD4 T cells (T cells), which can lead to AIDS (Acquired Immunodeficiency Syndrome) and AIDS (Acquired Immunodeficiency S